# DAA Kaggle Submission Notebook V.3

This notebook is TRYING to trains a multiclass model and TRYING to write `submission.csv` for the Kaggle DAA competition.

**IMPORTANT LOOK AT HERE AGAIN:** set `TRAIN_PATH` to the Kaggle training CSV. `dataset_test_X.csv` IS the provided test feature file. "IF" `sample_submission.csv` exists, the notebook uses its target column name and order.


In [1]:

import os, warnings, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

# ==== EDIT THESE PATHS IF NEEDED ====
TRAIN_PATH = "datasets/dataset_train.csv"              # e.g. "/kaggle/input/daa/train.csv"
TEST_PATH = "datasets/dataset_test_X.csv"      # provided file
SAMPLE_SUB_PATH = "sample_submission.csv"
SUB_PATH = "submission.csv"
TARGET = None                         # set manually if inference fails, like "age_group"
ID_COLS = ["PID", "HHID"]
# ====================================

test = pd.read_csv(TEST_PATH)
if not os.path.exists(TRAIN_PATH):
    raise FileNotFoundError("Set TRAIN_PATH to the Kaggle training CSV file.")
train = pd.read_csv(TRAIN_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH) if os.path.exists(SAMPLE_SUB_PATH) else None

# Infer target column
if TARGET is None:
    if sample_sub is not None:
        cand = [c for c in sample_sub.columns if c not in ID_COLS]
        if len(cand) == 1:
            TARGET = cand[0]
    if TARGET is None:
        for c in ["age_group", "AgeGroup", "AGE_GROUP", "age", "Age", "target", "Target", "class", "Class"]:
            if c in train.columns:
                TARGET = c
                break
    if TARGET is None:
        not_in_test = [c for c in train.columns if c not in test.columns and c not in ID_COLS]
        if len(not_in_test) == 1:
            TARGET = not_in_test[0]
    if TARGET is None:
        raise ValueError("Could not infer target column. Set TARGET manually.")

print("Target column:", TARGET)

# Features: columns present in both train and test, excluding IDs and target
feature_cols = [c for c in train.columns if c not in [TARGET] + ID_COLS and c in test.columns]
X = train[feature_cols].copy()
X_test = test[feature_cols].copy()

y_raw = train[TARGET].astype(str).copy()
test_ids = test["PID"] if "PID" in test.columns else pd.Series(np.arange(len(test)), name="PID")

le = LabelEncoder()
y = le.fit_transform(y_raw)

cat_cols = [c for c in feature_cols if X[c].dtype == "object" or str(X[c].dtype).startswith("category")]
for c in cat_cols:
    X[c] = X[c].fillna("MISSING").astype(str)
    X_test[c] = X_test[c].fillna("MISSING").astype(str)

print("N features:", len(feature_cols), "| categorical:", len(cat_cols))


Target column: AGE_RANGE
N features: 234 | categorical: 0


In [2]:
# Prefer CatBoost if available; otherwise fallback to sklearn HistGradientBoosting
try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except Exception:
    HAS_CATBOOST = False

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

# FIX: Dynamically find any column that is not a number (catches 'object' and 'string' types)
actual_cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Actually encoding {len(actual_cat_cols)} categorical columns.")

if HAS_CATBOOST:
    print("Using CatBoost")
    
    preprocessor = ColumnTransformer(
        transformers=[
            # Encode the actual string columns we just found
            ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), actual_cat_cols)
        ],
        remainder="passthrough" 
    )
    
    base = Pipeline(steps=[
        ("prep", preprocessor),
        ("clf", CatBoostClassifier(loss_function="MultiClass", random_state=RANDOM_STATE, verbose=False))
    ])
    
    param_dist = {
        "clf__iterations": [300, 600],
        "clf__depth": [4, 6, 8],
        "clf__learning_rate": [0.03, 0.05, 0.1],
        "clf__l2_leaf_reg": [1, 3, 10],
    }
    search = RandomizedSearchCV(
        base, param_dist, n_iter=8, scoring="f1_macro",
        cv=cv, n_jobs=-1, random_state=RANDOM_STATE, verbose=1
    )
    
    search.fit(X, y) 
    
    best_model = search.best_estimator_
    print("Best CV macro F1:", search.best_score_)
    pred_enc = np.asarray(best_model.predict(X_test)).astype(int).ravel()

else:
    print("CatBoost not available; using sklearn HistGradientBoosting fallback")
    
    num_cols = [c for c in feature_cols if c not in actual_cat_cols]

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ])
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
    ])
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_cols),
            ("cat", categorical_transformer, actual_cat_cols),
        ],
        remainder="drop"
    )

    base = Pipeline(steps=[
        ("prep", preprocessor),
        ("clf", HistGradientBoostingClassifier(random_state=RANDOM_STATE))
    ])

    param_dist = {
        "clf__learning_rate": [0.03, 0.05, 0.1],
        "clf__max_iter": [200, 400],
        "clf__max_leaf_nodes": [15, 31, 63],
        "clf__min_samples_leaf": [10, 20, 40],
        "clf__l2_regularization": [0, 0.1, 1.0],
    }
    search = RandomizedSearchCV(
        base, param_dist, n_iter=10, scoring="f1_macro",
        cv=cv, n_jobs=-1, random_state=RANDOM_STATE, verbose=1
    )
    search.fit(X, y)
    best_model = search.best_estimator_
    print("Best CV macro F1:", search.best_score_)
    pred_enc = best_model.predict(X_test).astype(int).ravel()

pred = le.inverse_transform(pred_enc)
print("Prediction distribution:")
print(pd.Series(pred).value_counts().head())

Actually encoding 229 categorical columns.
Using CatBoost
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best CV macro F1: 0.4508455794200037
Prediction distribution:
12    440
10    423
9     353
6     338
8     336
Name: count, dtype: int64


In [5]:

import os

# The Kaggle competition requires these exact column names
id_col = "RowId"
target_col = "AGE_RANGE"

# Check if sample_submission exists to get the exact IDs, otherwise use test['PID']
if os.path.exists("sample_submission.csv"):
    sample_sub = pd.read_csv("sample_submission.csv")
    sub = pd.DataFrame({
        id_col: sample_sub[id_col],
        target_col: pred
    })
elif "PID" in test.columns:
    # Fallback: Use PID from the test set
    sub = pd.DataFrame({
        id_col: test["PID"],
        target_col: pred
    })
else:
    raise ValueError("Could not find 'sample_submission.csv' or 'PID' column in test data.")

# Force the IDs to be integers (removes the .0)
sub[id_col] = sub[id_col].astype(int)

# Save the final submission
sub.to_csv("submission.csv", index=False)

print("Saved successfully without needing sample_submission.csv!")
print(sub.head())


Saved successfully without needing sample_submission.csv!
   RowId AGE_RANGE
0  11334         4
1    364         2
2  12938        10
3   4550         6
4   9729         9


In [6]:
# Create a DataFrame with purely sequential RowIds (0, 1, 2, ... 3654)
sub = pd.DataFrame({
    "RowId": range(len(pred)), 
    "AGE_RANGE": pred
})

# Save it
sub.to_csv("submission.csv", index=False)

print("Saved with correct sequential RowIds!")
print(sub.head())

Saved with correct sequential RowIds!
   RowId AGE_RANGE
0      0         4
1      1         2
2      2        10
3      3         6
4      4         9
